In [1]:
import asyncio
import time

from hchecks import (
    Probe,
    arun_check,
    arun_probe,
    HttpCheck,
    HostConnectivityCheck,
    CheckResult,
)

In [2]:
def sync_check() -> CheckResult:
    start = time.monotonic()
    time.sleep(1)
    duration_ms = (time.monotonic() - start) * 1000
    return CheckResult("SyncCheck", True, duration_ms=duration_ms)

In [3]:
async def async_check() -> CheckResult:
    start = time.monotonic()
    await asyncio.sleep(1)
    duration_ms = (time.monotonic() - start) * 1000
    return CheckResult("AsyncCheck", True, duration_ms=duration_ms)

In [4]:
async def araising_check() -> CheckResult:
    await asyncio.sleep(0.5)
    raise Exception("Asynchronous error")

In [5]:
def raising_check() -> CheckResult:
    time.sleep(1)
    raise Exception("Synchronous error")

In [6]:
await arun_check(sync_check)

CheckResult(name='SyncCheck', passed=True, details=None, duration_ms=1000.8632999961264)

In [7]:
await arun_check(async_check)

CheckResult(name='AsyncCheck', passed=True, details=None, duration_ms=1000.5173999961698)

In [8]:
probe = Probe("MyProbe",
    [
        HostConnectivityCheck("google.com", 443),
        HostConnectivityCheck("msn.com", 443),
        HttpCheck("https://github.com", timeout=3),
        sync_check,
        async_check,
        araising_check,
        raising_check,
    ],
    "A test probe"
)

In [9]:
result = await arun_probe(probe)
print("Name: ", result.name)
print("Description: ", result.description)
print("Duration, ms: ", result.duration_ms)
print("Healthy: ", result.healthy)
print("Checks:")
for r in result.checks:
    print(f"  {r}")

Name:  MyProbe
Description:  A test probe
Duration, ms:  1037.4308999889763
Healthy:  False
Checks:
  CheckResult(name='HostConnectivityCheck to google.com:443', passed=True, details=None, duration_ms=114.83740000403486)
  CheckResult(name='HostConnectivityCheck to msn.com:443', passed=True, details=None, duration_ms=83.5810999997193)
  CheckResult(name='HttpCheck to https://github.com', passed=True, details=None, duration_ms=626.5963000041666)
  CheckResult(name='SyncCheck', passed=True, details=None, duration_ms=1000.4309999931138)
  CheckResult(name='AsyncCheck', passed=True, details=None, duration_ms=1000.5872000037925)
  CheckResult(name='araising_check', passed=False, details='Exception: Asynchronous error', duration_ms=508.1579999969108)
  CheckResult(name='raising_check', passed=False, details='Exception: Synchronous error', duration_ms=1015.303699998185)


In [10]:
import json

print(json.dumps(result.dump(), indent=2))

{
  "name": "MyProbe",
  "description": "A test probe",
  "healthy": false,
  "duration_ms": 1037.4308999889763,
  "checks": [
    {
      "name": "HostConnectivityCheck to google.com:443",
      "passed": true,
      "details": null,
      "duration_ms": 114.83740000403486
    },
    {
      "name": "HostConnectivityCheck to msn.com:443",
      "passed": true,
      "details": null,
      "duration_ms": 83.5810999997193
    },
    {
      "name": "HttpCheck to https://github.com",
      "passed": true,
      "details": null,
      "duration_ms": 626.5963000041666
    },
    {
      "name": "SyncCheck",
      "passed": true,
      "details": null,
      "duration_ms": 1000.4309999931138
    },
    {
      "name": "AsyncCheck",
      "passed": true,
      "details": null,
      "duration_ms": 1000.5872000037925
    },
    {
      "name": "araising_check",
      "passed": false,
      "details": "Exception: Asynchronous error",
      "duration_ms": 508.1579999969108
    },
    {
      "n